
# UD4 — Redes Neuronales con PyTorch  
## Clasificación multiclase: **Fashion-MNIST completo**

En este notebook implementamos **el mismo problema multiclase** que ya resolvimos con Keras,
pero ahora usando **PyTorch**.

### Objetivo
- Entender:
  - logits vs probabilidades
  - `CrossEntropyLoss`
  - por qué **NO** aplicamos softmax explícito en el modelo


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import fashion_mnist
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay



## 1. Carga del dataset


In [ ]:

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()



## 2. Preprocesado

- Normalización
- Aplanado
- Etiquetas como enteros (NO one-hot)


In [ ]:

x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0
from torchvision import datasets, transforms
x_train = x_train.reshape(-1, 28*28)
x_test = x_test.reshape(-1, 28*28)

X_train = torch.tensor(x_train)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(x_test)
y_test_t = torch.tensor(y_test, dtype=torch.long)



## 3. Dataset y DataLoader


Breve explicación sobre `Dataset` y `DataLoader`:

- `Dataset`: interfaz que devuelve muestras y etiquetas. Usa `TensorDataset` para tensores en memoria o define una subclase personalizada para cargar imágenes/archivos y aplicar `transforms`.
- `DataLoader`: crea iteradores por batches. Ajusta `batch_size`, `shuffle`, `num_workers` y `pin_memory` según tu hardware.

Consejo práctico: con imágenes, utiliza `torchvision.datasets` + `transforms` y `num_workers>0` para mejorar throughput; para ejemplos sencillos `TensorDataset` + `DataLoader` es suficiente.


In [ ]:

train_ds = TensorDataset(X_train, y_train_t)
test_ds = TensorDataset(X_test, y_test_t)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128)


### Ejemplo: `Dataset` personalizado y `collate_fn` (multiclase)

Aquí mostramos cómo definir un `Dataset` que trabaja sobre arrays NumPy y una `collate_fn` adaptada a etiquetas multiclase (`long`). Útil para controlar la conversión y preparar batches complejos.


In [ ]:
from torch.utils.data import Dataset

class NumpyDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        if self.transform is not None:
            x = self.transform(x)
        return x, y

def multiclass_collate(batch):
    xs = np.stack([b[0] for b in batch], axis=0)
    xs = torch.tensor(xs, dtype=torch.float32)
    ys = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return xs, ys

# Uso
custom_ds = NumpyDataset(x_train, y_train)
custom_loader = DataLoader(custom_ds, batch_size=128, shuffle=True, collate_fn=multiclass_collate)
# sustituir el train_loader si se desea probar
train_loader = custom_loader


#### Ejemplo: `collate_fn` con padding (secuencias)

El siguiente `collate_fn` muestra cómo hacer padding para secuencias de distinta longitud y devolver también las longitudes reales por ejemplo para usar con RNNs.


In [ ]:
def pad_collate(batch):
    xs = [torch.tensor(b[0], dtype=torch.float32) for b in batch]
    lengths = torch.tensor([x.size(0) for x in xs], dtype=torch.long)
    maxlen = int(lengths.max())
    padded = torch.zeros(len(xs), maxlen, dtype=torch.float32)
    for i, x in enumerate(xs):
        padded[i, : x.size(0)] = x
    ys = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return padded, lengths, ys

# Prueba con datos sintéticos
import numpy as np
synthetic = [(np.random.rand(np.random.randint(3,8)), np.random.randint(0,10)) for _ in range(4)]
padded_x, lengths, y = pad_collate(synthetic)
print('padded_x.shape=', padded_x.shape, 'lengths=', lengths, 'y.shape=', y.shape)



## 4. Definición del modelo

Observa que **no usamos softmax aquí**.

Explicación breve: el modelo debe devolver *logits* (valores reales sin normalizar).
La función de pérdida `nn.CrossEntropyLoss` espera esos logits y aplica internamente
`log_softmax` + `NLLLoss` (es decir, softmax en log-espacio + la pérdida de verosimilitud negativa).
Esto es más eficiente y numéricamente estable que calcular `softmax` explícitamente en la red
antes de la pérdida. Si aplicases `softmax` en la salida del modelo, estarías pasando
probabilidades a la pérdida en lugar de logits (no es necesario y puede introducir problemas).

Nota práctica: para obtener probabilidades durante la inferencia usa `F.softmax(outputs, dim=1)`,
y para obtener la clase predicha basta `outputs.argmax(dim=1)` porque softmax no cambia el orden
de las puntuaciones (los logits).


In [ ]:

class MultiClassNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)  # logits
        return x

model = MultiClassNet()
model



## 5. Función de pérdida y optimizador

`CrossEntropyLoss` incluye internamente `log_softmax` + `NLLLoss`.

Por qué importa: en lugar de calcular `softmax` y luego la entropía cruzada por separado,
PyTorch combina ambas operaciones en una implementación numéricamente estable. Esto evita
problemas de underflow/overflow y suele dar gradientes más fiables durante el entrenamiento.

Equivalencias útiles: para clasificación multiclase en Keras se suele usar `activation='softmax'`
en la última capa + `categorical_crossentropy`. En PyTorch normalmente dejamos la última capa
sin activar (logits) y usamos `nn.CrossEntropyLoss()`.


In [ ]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)



## 6. Bucle de entrenamiento


In [ ]:

epochs = 15
train_losses = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")



## 7. Evaluación


In [ ]:

model.eval()
correct = 0
total = 0
all_preds = []
all_true = []

with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb)  # logits: salidas sin normalizar (raw scores)
        # Para la predicción de clase no hace falta softmax: basta argmax sobre logits
        preds = torch.argmax(logits, dim=1)
        # Si quieres probabilidades explícitas (p. ej. para mostrar) usa softmax en inferencia:
        # probs = F.softmax(logits, dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
        all_preds.extend(preds.numpy())
        all_true.extend(yb.numpy())

print("Test accuracy:", correct / total)



## 8. Curva de pérdida


In [ ]:

plt.plot(train_losses)
plt.title("Training loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()



## 9. Matriz de confusión


In [ ]:

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

cm = confusion_matrix(all_true, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(xticks_rotation=45)
plt.show()



## 10. Conclusiones

- PyTorch exige entender logits
- La pérdida gestiona softmax internamente
- El flujo es completamente explícito

En el apéndice veremos **JAX** y luego una **comparativa final**.
